# ⚡ IndexTTS-2.5 Audiobook Instant Runner (Google Colab)
這是 **index-tts-audiobook** 專為 Google Colab 設計的極簡快速版筆記本。

### 特色：
- **極簡兩步驟**：步驟 1 裝好環境，步驟 2 全自動掛載 Drive、下載 IndexTTS-2.5 完整模型 (含 `codec.pth`) 並直接合成音訊！
- **斷點續傳**：每一段落獨立存檔，斷線重新執行不重複運算。
- **A100 / L4 / T4 GPU 加速**：自動偵測 CUDA 進行高效推論。


## 步驟 1：檢查 GPU 並安裝所有相依環境 (約 1 分鐘)
確保 GPU 正常，並安裝 IndexTTS-2.5 與 audiobook-pipeline 所需的完整套件。


In [ ]:
# 1. 檢查 GPU
!nvidia-smi

# 2. 下載 IndexTTS 程式庫
import os
%cd /content
if not os.path.exists("/content/index-tts"):
    !git clone --depth 1 https://github.com/index-tts/index-tts.git /content/index-tts

%cd /content/index-tts
# 3. 移除衝突的預裝 tensorflow，並安裝正確版本套件
!pip uninstall -y -q tensorflow
!pip install -q -U "transformers==4.52.1" "accelerate==1.8.1" "tokenizers==0.21.0"
!pip install -q -U openai-whisper descript-audiotools cn2an g2p-en WeTextProcessing
!pip install -q -U munch omegaconf einops json5 textstat pydub sentencepiece safetensors librosa jieba
!pip install -q opencc-python-reimplemented soundfile huggingface_hub modelscope

# 4. 安裝最新 audiobook-pipeline
!pip install -q --upgrade git+https://github.com/HOWARD1021/index-tts-audiobook.git

print("✅ GPU 與所有相依環境安裝完成！")


## 步驟 2：⚡ 一鍵即時跑 (全自動掛載、下載 2.5 完整權重並開始合成)
點擊執行後會自動：
1. 掛載 Google Drive 工作區 (`/content/drive/MyDrive/audiobook-workspace`)
2. 檢查並自動從 Hugging Face 下載 IndexTTS-2.5 完整模型 (含 `codec.pth`，約 25 秒)
3. 自動生成 CUDA 設定檔與示範文稿（若尚無文稿）
4. 自動讀取 `prompts/voice.wav` 開始 GPU 合成！


In [ ]:
from google.colab import drive
from pathlib import Path
import os
from huggingface_hub import snapshot_download

# 1. 掛載 Google Drive
drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive/audiobook-workspace")
CHECKPOINTS_DIR = DRIVE_ROOT / "checkpoints" / "IndexTTS-2.5"
PROMPTS_DIR = DRIVE_ROOT / "prompts"
SCRIPTS_DIR = DRIVE_ROOT / "scripts"
OUTPUT_DIR = DRIVE_ROOT / "output"
CONFIG_DIR = DRIVE_ROOT / "config"

for p in [CHECKPOINTS_DIR, PROMPTS_DIR, SCRIPTS_DIR, OUTPUT_DIR, CONFIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# 2. 自動檢查並下載 IndexTTS-2.5 官方權重 (含 codec.pth)
required_files = ["config.yaml", "codec.pth", "gpt.pth", "s2mel.pth", "wav2vec2bert_stats.pt"]
if not all((CHECKPOINTS_DIR / f).exists() for f in required_files):
    print("⏳ 正在下載 IndexTTS-2.5 完整模型權重至 Google Drive（約 4.3 GB）...")
    snapshot_download(
        repo_id="IndexTeam/IndexTTS-2.5",
        local_dir=str(CHECKPOINTS_DIR),
        local_dir_use_symlinks=False,
    )
    print("✅ IndexTTS-2.5 模型就緒！")
else:
    print("✅ IndexTTS-2.5 模型已就緒！")

# 3. 確保 CUDA 配置檔存在
config_path = CONFIG_DIR / "colab-cuda.toml"
if not config_path.exists():
    config_path.write_text("""[defaults]
language = "ZH"
device = "cuda"
max_chunk_chars = 400
max_text_tokens_per_segment = 100
interval_silence_ms = 250
inter_chunk_pause_ms = 450
emotion_span_pause_ms = 80
text_normalization = true
use_random = false
use_qwen_emo = false
sample_rate = 22050
channels = 1
max_seconds_per_char = 0.8
max_mel_tokens = 800
temperature = 1.0
top_k = 30
top_p = 0.8
repetition_penalty = 10.0
speed = 1.0
seed = 42
memory_limit_gb = 16.0

[defaults.emotion]
vector = [0.30, 0.0, 0.0, 0.0, 0.0, 0.0, 0.15, 0.35]
alpha = 1.0
bold_vector = [0.45, 0.0, 0.0, 0.0, 0.0, 0.20, 0.10]
bold_alpha = 1.0
italic_vector = [0.15, 0.0, 0.0, 0.0, 0.0, 0.20, 0.0, 0.45]
italic_alpha = 1.0
""", encoding="utf-8")

# 4. 確保文稿存在
sample_script = SCRIPTS_DIR / "sample-chapter.md"
prep_script = SCRIPTS_DIR / "sample-chapter-simplified.md"
if not prep_script.exists():
    if not sample_script.exists():
        sample_script.write_text("# 第一章：啟程\n\n這是一個寧靜的早晨，陽光穿透薄霧，灑在青石街道上。旅人背起行囊，準備迎接未知的冒險。\n\n**「這條路將會通向何方？」** 他心中自問，步伐卻顯得堅定無比。\n", encoding="utf-8")
    os.system(f'audiobook prepare --input "{sample_script}" --output "{prep_script}"')

# 5. 確認參考音訊
prompt_wavs = list(PROMPTS_DIR.glob("*.wav"))
if not prompt_wavs:
    raise FileNotFoundError(f"找不到參考聲音，請確認已放入：{PROMPTS_DIR}")
selected_prompt = prompt_wavs[0]
output_wav = OUTPUT_DIR / "sample-chapter.wav"

print(f"🎙️ 參考聲音: {selected_prompt.name}")
print(f"🎯 輸出路徑: {output_wav}")
print("🚀 開始在 CUDA GPU 進行 IndexTTS-2.5 合成...")

# 6. 直接開始 GPU 合成
render_cmd = (
    f"USE_TF=0 audiobook render "
    f"--backend indextts-2.5 "
    f"--script \"{prep_script}\" "
    f"--output \"{output_wav}\" "
    f"--project-root \"/content/index-tts\" "
    f"--model-dir \"{CHECKPOINTS_DIR}\" "
    f"--prompt \"{selected_prompt}\" "
    f"--config \"{config_path}\" "
    f"--device cuda"
)

!{render_cmd}


## 步驟 3：音訊品質驗證與線上試聽
驗證產出的音訊規格 (PCM 16-bit, 22,050 Hz, 單聲道)，並直接在瀏覽器中播放試聽。


In [ ]:
import json
import soundfile as sf
from IPython.display import Audio, display
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/audiobook-workspace")
OUTPUT_DIR = DRIVE_ROOT / "output"
CONFIG_DIR = DRIVE_ROOT / "config"
config_path = CONFIG_DIR / "colab-cuda.toml"
output_wav = OUTPUT_DIR / "sample-chapter.wav"
manifest_path = OUTPUT_DIR / "sample-chapter.manifest.json"

# 1. 執行管線品質標準檢驗
!audiobook validate --wav "{output_wav}" --backend indextts-2.5 --config "{config_path}"

# 2. 顯示生成統計
if manifest_path.exists():
    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)
    chunks = manifest.get("chunks", [])
    duration = manifest.get("final_duration_seconds", 0)
    print(f"\n📊 章節摘要:")
    print(f"  狀態:          {manifest.get('status')}")
    print(f"  合成段落數:    {len(chunks)}")
    print(f"  音訊總長度:    {duration:.2f} 秒 ({duration/60:.2f} 分鐘)")

# 3. 瀏覽器播放器
if output_wav.exists():
    data, sr = sf.read(str(output_wav))
    print(f"\n▶️ 線上試聽 ({sr} Hz, {len(data)} 取樣點):")
    display(Audio(data, rate=sr))


## 步驟 4：批次合成整本書（13 章一次跑完）
只要把章節 Markdown（例如 `00-foreword-zh-simplified.md` 等）放入 `scripts/`，執行此儲存格即可依序自動合成整部有聲書！


In [ ]:
from pathlib import Path
import os

DRIVE_ROOT = Path("/content/drive/MyDrive/audiobook-workspace")
CHECKPOINTS_DIR = DRIVE_ROOT / "checkpoints" / "IndexTTS-2.5"
PROMPTS_DIR = DRIVE_ROOT / "prompts"
SCRIPTS_DIR = DRIVE_ROOT / "scripts"
OUTPUT_DIR = DRIVE_ROOT / "output"
CONFIG_DIR = DRIVE_ROOT / "config"
config_path = CONFIG_DIR / "colab-cuda.toml"

raw_chapters = sorted([
    p for p in SCRIPTS_DIR.glob("*.md")
    if not p.name.endswith("-simplified.md") and not p.name.startswith("sample-")
])

prompt_wavs = list(PROMPTS_DIR.glob("*.wav"))
if not prompt_wavs:
    raise FileNotFoundError(f"缺少 prompts WAV 於 {PROMPTS_DIR}")
selected_prompt = prompt_wavs[0]

print(f"📚 找到 {len(raw_chapters)} 個章節準備批次合成:")
for ch in raw_chapters:
    print(f"  - {ch.name}")

for ch in raw_chapters:
    stem = ch.stem
    prep_path = SCRIPTS_DIR / f"{stem}-simplified.md"
    out_wav = OUTPUT_DIR / f"{stem}.wav"
    
    print(f"\n=======================================================")
    print(f"🚀 正在批次合成: {stem}")
    print(f"=======================================================")
    
    # 1. 簡化前處理
    prepare_cmd = f'audiobook prepare --input "{ch}" --output "{prep_path}"'
    !{prepare_cmd}
    
    # 2. CUDA 合成
    render_cmd = (
        f"USE_TF=0 audiobook render "
        f"--backend indextts-2.5 "
        f"--script \"{prep_path}\" "
        f"--output \"{out_wav}\" "
        f"--project-root \"/content/index-tts\" "
        f"--model-dir \"{CHECKPOINTS_DIR}\" "
        f"--prompt \"{selected_prompt}\" "
        f"--config \"{config_path}\" "
        f"--device cuda"
    )
    !{render_cmd}
      
    # 3. 品質驗證
    val_cmd = f'audiobook validate --wav "{out_wav}" --backend indextts-2.5 --config "{config_path}"'
    !{val_cmd}

print("\n🎉 所有章節批次合成完成！成果皆已安全保存在 Google Drive 中。")
